# MobileADAS3D-H2 single-image capacity gate

Run exactly 1,000 optimizer steps on the same audited KITTI image containing Vehicle and Pedestrian. This tests whether spatial reference queries preserve H1-v2's local capacity before Tiny16. It is not a benchmark. Distillation is disabled.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from collections import deque
import hashlib,json,os,shlex,subprocess,sys
REPO_URL='https://github.com/Ali-RT/mobile_adas3d.git'; BRANCH='main'
PROJECT_DIR=Path('/content/mobile_adas3d')
DRIVE_DATASET_ROOT=Path('/content/drive/MyDrive/datasets/kitti')
LOCAL_DATASET_ROOT=Path('/content/kitti'); DATASET_VIEW=Path('/content/kitti_h1')
FULL_SPLIT_DIR=Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
SINGLE_SPLIT_DIR=Path('/content/drive/MyDrive/mobile_adas3d_splits/h2_single')
OUTPUT_DIR=Path('/content/drive/MyDrive/mobile_adas3d_outputs/mobileadas3d_h2_single')
CONFIG=PROJECT_DIR/'configs/kitti_mobileadas3d_h2_single_overfit.yaml'
CHECKPOINT=OUTPUT_DIR/'single_image_latest.pt'
def run(command,cwd=None):
    command=[str(x) for x in command]; print('+',shlex.join(command),flush=True)
    result=subprocess.run(command,cwd=cwd)
    if result.returncode: raise RuntimeError(f'Exit {result.returncode}: {shlex.join(command)}')
def run_streamed(command,cwd,log_path,allow_failure=False):
    command=[str(x) for x in command]; log_path.parent.mkdir(parents=True,exist_ok=True)
    print('+',shlex.join(command),flush=True); print('Durable log:',log_path,flush=True)
    tail=deque(maxlen=160); env=os.environ.copy(); env['PYTHONUNBUFFERED']='1'
    with log_path.open('w',encoding='utf-8',buffering=1) as log:
        p=subprocess.Popen(command,cwd=cwd,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout: print(line,end='',flush=True); log.write(line); tail.append(line.rstrip())
        code=p.wait()
    if code and not allow_failure: raise RuntimeError(f'Exit {code}; full log={log_path}\n'+'\n'.join(tail))
    return code


In [ ]:
# Refresh code and require CUDA. The H2 workflow commit must be available remotely first.
if not (PROJECT_DIR/'.git').exists(): run(['git','clone','--branch',BRANCH,REPO_URL,PROJECT_DIR])
else:
    run(['git','fetch','origin'],PROJECT_DIR); run(['git','checkout',BRANCH],PROJECT_DIR); run(['git','pull','--ff-only','origin',BRANCH],PROJECT_DIR)
os.chdir(PROJECT_DIR); run([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'],PROJECT_DIR)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Choose Runtime > Change runtime type > GPU')
print('Commit:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=PROJECT_DIR,text=True).strip())
print('GPU:',torch.cuda.get_device_name(0))


In [ ]:
# Resolve KITTI and select the same deterministic two-class Chen-train image used by H1.
def count_files(path,suffix): return sum(1 for p in path.iterdir() if p.is_file() and p.suffix==suffix) if path.is_dir() else 0
def complete(root): return count_files(root/'training/image_2','.png')==7481 and count_files(root/'training/label_2','.txt')==7481 and count_files(root/'training/calib','.txt')==7481
if complete(LOCAL_DATASET_ROOT): DATASET_ROOT=LOCAL_DATASET_ROOT
else:
    aliases={'image_2':['image_2','image_02'],'label_2':['label_2','label_02'],'calib':['calib']}
    (DATASET_VIEW/'training').mkdir(parents=True,exist_ok=True)
    for canonical,candidates in aliases.items():
        source=next((DRIVE_DATASET_ROOT/'training'/name for name in candidates if (DRIVE_DATASET_ROOT/'training'/name).is_dir()),None)
        if source is None: raise FileNotFoundError(f'Missing source for {canonical}')
        link=DATASET_VIEW/'training'/canonical
        if not link.exists() and not link.is_symlink(): link.symlink_to(source,target_is_directory=True)
    DATASET_ROOT=DATASET_VIEW
if not complete(DATASET_ROOT): raise RuntimeError(f'KITTI view incomplete: {DATASET_ROOT}')
full_ids=[x.strip() for x in (FULL_SPLIT_DIR/'train.txt').read_text().splitlines() if x.strip()]
vehicle={'Car','Van','Truck','Tram'}; pedestrian={'Pedestrian','Person_sitting'}
SAMPLE_ID=None
for sample_id in full_ids:
    names={line.split()[0] for line in (DATASET_ROOT/'training/label_2'/f'{sample_id}.txt').read_text().splitlines() if line.strip()}
    if names & vehicle and names & pedestrian: SAMPLE_ID=sample_id; break
if SAMPLE_ID is None: raise RuntimeError('No Chen-train sample contains both product classes')
COMPARISON_ID=next(x for x in full_ids if x!=SAMPLE_ID)
payload=SAMPLE_ID+'\n'; SINGLE_SPLIT_DIR.mkdir(parents=True,exist_ok=True)
for name in ('train.txt','val.txt'): (SINGLE_SPLIT_DIR/name).write_text(payload)
manifest={'schema_version':1,'purpose':'MobileADAS3D-H2 single-image capacity only','architecture':'MobileADAS3D-H2','sample_id':SAMPLE_ID,'comparison_id':COMPARISON_ID,'split_sha256':hashlib.sha256(payload.encode()).hexdigest(),'distillation_enabled':False}
(SINGLE_SPLIT_DIR/'single_image_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
print(json.dumps(manifest,indent=2)); print('Dataset root:',DATASET_ROOT)


In [ ]:
# Validate H2, the unchanged H1-v2 objective, and a real CUDA loss path.
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
COMMON=['--profile','colab_drive','--dataset-root',DATASET_ROOT,'--split-dir',SINGLE_SPLIT_DIR,'--output-dir',OUTPUT_DIR]
run([sys.executable,'-m','unittest','discover','-s','tests','-p','test_mobileadas3d_h2.py','-v'],PROJECT_DIR)
run([sys.executable,'-m','unittest','discover','-s','tests','-p','test_h1_set_training.py','-v'],PROJECT_DIR)
run_streamed([sys.executable,'-u','scripts/check_training_ready.py','--config',CONFIG,*COMMON,'--require-cuda','--report',OUTPUT_DIR/'training_preflight.json'],PROJECT_DIR,OUTPUT_DIR/'training_preflight.log')
from tools.config import load_config
cfg=load_config(str(CONFIG))
assert cfg['model']['name']=='MobileADAS3D-H2' and cfg['model']['center_offset_scale']==0.10
assert cfg['loss']['classification_mode']=='implicit_background_softmax' and cfg['distillation']['enabled'] is False
print('H2 single-image preflight passed; distillation=false')


In [ ]:
# Exactly 1,000 optimizer steps. The atomic checkpoint is isolated from H1 and auto-resumed.
TRAIN_REPORT=OUTPUT_DIR/'single_image_training_report.json'
run_streamed([sys.executable,'-u','scripts/run_h1_single_image_overfit.py','--config',CONFIG,*COMMON,'--steps','1000','--save-interval','100','--log-interval','10','--checkpoint',CHECKPOINT,'--report',TRAIN_REPORT],PROJECT_DIR,OUTPUT_DIR/'single_image_training.log')
training_report=json.loads(TRAIN_REPORT.read_text())
if training_report.get('architecture')!='MobileADAS3D-H2': raise RuntimeError(training_report)
print(json.dumps(training_report,indent=2))


In [ ]:
# Apply the unchanged query-quality and cross-image sensitivity gates.
QUERY_REPORT=OUTPUT_DIR/'single_image_query_diagnostics.json'
SENSITIVITY_REPORT=OUTPUT_DIR/'single_image_sensitivity.json'
query_code=run_streamed([sys.executable,'-u','scripts/diagnose_h1_queries.py','--config',CONFIG,*COMMON,'--checkpoint',CHECKPOINT,'--split','val','--score-threshold','0.1','--report',QUERY_REPORT],PROJECT_DIR,OUTPUT_DIR/'single_image_query_diagnostics.log',allow_failure=True)
sensitivity_code=run_streamed([sys.executable,'-u','scripts/diagnose_h1_image_sensitivity.py','--config',CONFIG,'--profile','colab_drive','--dataset-root',DATASET_ROOT,'--output-dir',OUTPUT_DIR,'--checkpoint',CHECKPOINT,'--sample-a',SAMPLE_ID,'--sample-b',COMPARISON_ID,'--report',SENSITIVITY_REPORT],PROJECT_DIR,OUTPUT_DIR/'single_image_sensitivity.log',allow_failure=True)
print('QUERY REPORT\n',QUERY_REPORT.read_text())
print('SENSITIVITY REPORT\n',SENSITIVITY_REPORT.read_text())
if query_code or sensitivity_code: print('STOP: send both reports; do not run Tiny16, full KITTI, or distillation.')
else: print('PASS: send both reports before preparing H2 Tiny16.')
